In [ ]:
#pip install sentence-transformers scikit-learn pandas numpy matplotlib hdbscan

In [ ]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
import hdbscan

In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv('test.csv', sep=';', encoding='windows-1251')
df

,label,query
0,теоретический вопрос,почему l1 дает разреженные веса
1,теоретический вопрос,разница между accuracy и F1
2,теоретический вопрос,в чем суть метода k-ближайших соседей
3,теоретический вопрос,зачем использовать ансамбли моделей
4,теоретический вопрос,что такое корреляция спирмена и пирсона разница
...,...,...
394,чушь,В стоматологии обязательно ли проводить извлеч...
395,чушь,Сколько слоев термопасты нужно наносить на кры...
396,чушь,подключи модель к самой себе
397,чушь,Отсортируй строки по степени их уверенности в ...


In [ ]:
df = pd.read_csv('train.csv', sep=';', encoding='windows-1251')

# Split the 'label;query' column into 'label' and 'query' columns
#df[['label', 'query']] = df['label;query'].str.split(';', n=1, expand=True)

# Now extract texts from the new 'query' column
texts = df['query'].astype(str).tolist()
print("Total samples:", len(texts))

Total samples: 1608


In [ ]:
df

,label,query
0,анализ данных,сколько строк в категории 'test'
1,анализ данных,найди топ-10 самых дорогих товаров в таблице
2,анализ данных,сделай частотный анализ для текстовой колонки
3,анализ данных,какие типы данных у нас в датасете
4,анализ данных,чекни на нуллы
...,...,...
1603,взаимодействие с графом,удали весь граф
1604,анализ данных и взаимодействие с графом,"В данных сильная асимметрия распределений, доб..."
1605,анализ данных и взаимодействие с графом,"Модель часто ошибается на крайних значениях, д..."
1606,анализ данных и взаимодействие с графом,"Модель путает классы, добавь балансировку или ..."


In [ ]:
print("Количество NaN по столбцам:")
print(df.isna().sum())

# 2. Столбцы, в которых есть хотя бы один NaN
print("\nСтолбцы с NaN:")
print(df.columns[df.isna().any()].tolist())

# 3. Строки, содержащие хотя бы один NaN (индексы)
print("\nИндексы строк, где есть NaN:")
print(df[df.isna().any(axis=1)].index.tolist())

# 4. Показать все строки с любым NaN (первые 5, чтобы не перегружать вывод)
print("\nПримеры строк с NaN (первые 5):")
print(df[df.isna().any(axis=1)].head())


Количество NaN по столбцам:
label    0
query    0
dtype: int64

Столбцы с NaN:
[]

Индексы строк, где есть NaN:
[]

Примеры строк с NaN (первые 5):
Empty DataFrame
Columns: [label, query]
Index: []


In [ ]:
model = SentenceTransformer("sentence-transformers/LaBSE", trust_remote_code=True)

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.array(embeddings)

print("Embedding shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

Batches:   0%|          | 0/51 [00:00<?, ?it/s]

Embedding shape: (1608, 768)


In [ ]:
def semantic_dedup_with_logging(texts, embeddings, threshold):
    kept_indices = []

    removed = []  # сюда будем складывать удалённые

    for i in range(len(texts)):
        if len(kept_indices) == 0:
            kept_indices.append(i)
            continue

        sims = cosine_similarity(
            [embeddings[i]],
            embeddings[kept_indices]
        )[0]

        max_sim_idx = np.argmax(sims)
        max_sim = sims[max_sim_idx]

        if max_sim < threshold:
            kept_indices.append(i)
        else:
            similar_to_idx = kept_indices[max_sim_idx]

            removed.append({
                "removed_text": texts[i],
                "kept_text": texts[similar_to_idx],
                "similarity": float(max_sim)
            })

    return kept_indices, removed

In [ ]:
kept_idx, removed_pairs = semantic_dedup_with_logging(
    texts,
    embeddings,
    threshold=0.80
)

dedup_texts = [texts[i] for i in kept_idx]

print("До:", len(texts))
print("После:", len(dedup_texts))
print("Удалено:", len(removed_pairs))

До: 1608
После: 1527
Удалено: 81


In [ ]:
def show_removed(removed_pairs, n):
    for i, r in enumerate(removed_pairs[:n]):
        print(f"\n--- {i+1} ---")
        print("Удалён:", r["removed_text"])
        print("Похож на:", r["kept_text"])
        print("Similarity:", round(r["similarity"], 3))


show_removed(removed_pairs, n=100)


--- 1 ---
Удалён: покажи первые 30 строк
Похож на: покажи первые 20 записей
Similarity: 0.844

--- 2 ---
Удалён: выведи данные о датасете
Похож на: выведи инфо о датафрейме
Similarity: 0.881

--- 3 ---
Удалён: посчитай стандартную ошибку среднего
Похож на: посчитай стандартную ошибку для прогноза
Similarity: 0.818

--- 4 ---
Удалён: визуализируй важность признаков
Похож на: покажи важность признаков через feature_importance
Similarity: 0.8

--- 5 ---
Удалён: визуализируй пропуски в данных
Похож на: покажи инфо о пропусках в данных
Similarity: 0.854

--- 6 ---
Удалён: посчитай количество пропусков в каждой колонке по отдельности
Похож на: Посчитай количество пропусков в каждом столбце
Similarity: 0.886

--- 7 ---
Удалён: выведи топ 5 самых коррелирующих признаков
Похож на: Покажи топ-5 признаков по влиянию
Similarity: 0.805

--- 8 ---
Удалён: сделай heatmap корреляций
Похож на: построй heatmap матрицы корреляций
Similarity: 0.877

--- 9 ---
Удалён: проверь типы данных во всех колонках


In [ ]:
# Создаём DataFrame только из сохранённых строк
df_deduplicated = df.iloc[kept_idx].copy()

# Сохраняем в файл (разделитель как в исходном – точка с запятой)
df_deduplicated.to_csv('train_deduplicated.csv', sep=';', encoding='utf-8', index=False)

print(f"Размер нового датасета: {len(df_deduplicated)} строк")
print(f"Сохранён в 'train_deduplicated.csv'")

Размер нового датасета: 1527 строк
Сохранён в 'train_deduplicated.csv'
